In [19]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent , AgentExecutor
from langchain_classic import hub
from langchain_core.prompts import PromptTemplate

In [2]:
load_dotenv()

True

In [ ]:
search_tool=DuckDuckGoSearchRun()

In [31]:
@tool
def get_weather_data(city:str)->str:
    """This Function fetches the current weather data for a given city """
    url=f'https://api.weatherstack.com/current?access_key=5f88da202aa6fd116253a6b2381e51db&query={city}'
    response=requests.get(url)
    return response.json()

In [32]:
llm=HuggingFaceEndpoint(model='Qwen/Qwen3-32B',
                        task='text_generation')
model=ChatHuggingFace(llm=llm)


In [33]:
template="""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt=PromptTemplate(template=template,
                      input_variables=['input', 'tools','tool_names','agent_scratchpad'])

In [34]:
#Step-3 : create react agent manually
agent=create_react_agent(
    llm=model,
    tools=[get_weather_data],
    prompt=prompt   
)

In [35]:
# Step-4 :Wrap it with AgentExecutor 
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True 
)

In [36]:
# Step5 :invoke
response=agent_executor.invoke({'input':'Find the capital of Madhya pradesh ,then find its current weather condition'})
print(response)




> Entering new AgentExecutor chain...
<think>
Okay, I need to answer the question: Find the capital of Madhya Pradesh, then find its current weather condition.

First, the user is asking for two things. The first part is to find the capital of Madhya Pradesh. I should recall that Madhya Pradesh is a state in India. From what I remember, the capital of Madhya Pradesh is Bhopal. But wait, I should confirm that because sometimes there can be confusion with other cities. Let me think—yes, Bhopal is definitely the capital. 

So once I have the capital, the next step is to get the current weather conditions for Bhopal. The user provided a tool called get_weather_data, which takes a city name as input. Since I know the capital is Bhopal, I should use that tool with "Bhopal" as the action input. 

Wait, but what if I'm wrong about Bhopal being the capital? Should I double-check that before proceeding? The problem is, I don't have a tool to look up the capital of a state. But I'm pretty confi